# Pipeline v17 - Merged SCS + STA Connectivity

| Branch | Recording phase | Detection unit | Statistical test |
|--------|-----------------|----------------|------------------|
| SCS    | Spontaneous protocol step | Discrete EPSP / IPSP events | Binomial score table |
| STA    | Stimulated protocol step | Spike-triggered average | Welch t-test |

**Per-connection consensus tier:** `both`, `scs_only`, `sta_only`, `conflict`, `unvalidated`.
**Per-neuron classification:** `confident_<exc|inh>`, `putative_<exc|inh>`, `ambiguous`, `no_valid_outgoing_connections`.

Inputs come from `SCS_MovieSplit_v17.m`, which walks each savefast's protocol
steps and writes only the CSVs an FOV has data for: `_spont_part{1,2}.csv`
when a spontaneous step is present, `_stim_part{1,2}.csv` when a stim step
is present, plus a `_stim_meta.json` carrying INCORRECT stim mask, stim frames.

Major updates from v16: applies SourceCorrection step at CSV generation in MATLAB

## 0. User Settings

In [ ]:
import os
from datetime import datetime
import matplotlib.pyplot as plt
plt.close('all')

### If fresh analysis, define plate and desired output path below. 
### If re-analyzing from pickle, set path in section 6b and LOAD_FROM_CACHE & RELOAD_DATAFRAMES to True in section 0

In [ ]:
#### Paths & mode ##########################################################################################
INPUT_DIR    = r'R:\QNP\2026_QNP\2026-05-19_JenniferGrooms_FireflyOne\PlateNumber_2_forGrace\v17_traces'
SELECTED_FOV = 'FOV_0003'   # used when MODE = 'single' and for inspect after batch
MODE         = 'batch'     # 'single' or 'batch'

# ### Output directory: one v17 folder per plate ###########################################
_LOCAL_ROOT = r'D:\Projects_Analysis\SCS'
_stamp = datetime.now().strftime('%Y%m%d_%H%M')
_rel = os.path.relpath(INPUT_DIR, r'R:\QNP\2026_QNP')
OUTPUT_DIR = os.path.join(_LOCAL_ROOT, _rel.replace('v17_traces', f'v17_outputs_{_stamp}'))
# old OUTPUT_DIR = INPUT_DIR.replace('v17_traces', f'v17_outputs_{_stamp}')
PLATE_NUMBER = None

# ### Per-method thresholds (per spec - keep distinct for now) #####################
SCS_SCORE_THRESH = 1.7
SCS_MIN_AP       = 2
STA_SCORE_THRESH = 1.3
STA_MIN_AP       = 2

# ### Metadata Excel Sheet file paths - add new rounds here ############
# add new rounds to experiment_mapping.csv for the following cell where metadata is selected
METADATA_PATHS = {
    'QNP-022': r'S:\QNP\2026_QNP\2026-02-18_JenniferGrooms_FireflySix_QNP-022\QNP-022_expMetadata.xlsx',
    'QNP-033': r'S:\QNP\2026_QNP\2026-03-16_JenniferGrooms_FireflyOne_QNP-033\QNP-033_expMetadata.xlsx',
    'QNP-036': r'S:\QNP\2026_QNP\2026-03-25_JenniferGrooms_FireflyOne_QNP-036\QNP-036_expMetadata.xlsx',
    'QNP-038': r'S:\QNP\2026_QNP\2026-03-25_JenniferGrooms_FireflyOne_QNP-036\QNP-038_expMetadata.xlsx',
    'QNP-047': r'S:\QNP\2026_QNP\2026-04-13_JenniferGrooms_FireflyOne_QNP-047\QNP-047_expMetadata.xlsx',
    'QNP-051': r'S:\QNP\2026_QNP\2026-04-22_JenniferGrooms_FireflyOne_QNP-051\QNP-051_expMetadata.xlsx',
    'QNP-057': r'S:\QNP\2026_QNP\2026-05-01_JenniferGrooms_FireflyOne_QNP-057\QNP-057_expMetadata.xlsx',
    'QNP-064': r'S:\QNP\2026_QNP\2026-05-06_JenniferGrooms_FireflyOne_QNP-064\QNP-064_expMetadata.xlsx',
    'QNP-066': r'S:\QNP\2026_QNP\2026-05-07_JenniferGrooms_FireflyOne_QNP-066\QNP-066_expMetadata.xlsx',
    'QNP-068': r'S:\QNP\2026_QNP\2026-05-12_JenniferGrooms_FireflyOne_QNP-068\QNP-068_expMetadata_.xlsx',
    'QNP-069': r'S:\QNP\2026_QNP\2026-05-19_JenniferGrooms_FireflyOne_QNP-069\QNP-069_expMetadata_.xlsx',
}

DISPLAY_NAMES = {
      'confident_exc':                 'Excitatory',
      'putative_exc':                  'Excitatory',
      'confident_inh':                 'Inhibitory',
      'putative_inh':                  'Inhibitory',
      'ambiguous':                     'Ambiguous',
      'no_valid_outgoing_connections': 'No validated efferent connections',
  }


# LOAD_FROM_CACHE   : set True only when you want to reload from disk.
#                     Default False so a full notebook run never overwrites live results.
# RELOAD_DATAFRAMES : set True to also reload df_all_conn_md / df_all_neurons from CSV.
#                     Leave False if those are already in memory from a live batch.
LOAD_FROM_CACHE   = False
RELOAD_DATAFRAMES = False

In [ ]:
import re, io, pandas as pd
QNP_ROUND = None
_plate_match = re.search(r'PlateNumber_(\d+)', INPUT_DIR)
PLATE_NUMBER = int(_plate_match.group(1)) if _plate_match else PLATE_NUMBER

_date_match = re.search(r'(\d{4}-\d{2}-\d{2})', INPUT_DIR)
_date = _date_match.group(1) if _date_match else None

if QNP_ROUND is None and _date is not None:
    with open(r'S:\QNP\2026_QNP\experiment_mapping.csv', encoding='utf-8-sig') as _f:
        _content = _f.read().replace('"', '')
    _mapping = pd.read_csv(io.StringIO(_content))
    _row = _mapping[_mapping['date'] == _date]
    QNP_ROUND = _row['qnp_round'].iloc[0] if not _row.empty else None

print(f'QNP_ROUND={QNP_ROUND}  PLATE_NUMBER={PLATE_NUMBER}')

## 1. Imports & Auto-discovery

In [ ]:
import os, re, sys, glob, time, json, traceback, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import utils.quiver_style
import utils.visualization
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__' if '__file__' in dir() else '.')))
sys.path.insert(0, '.')
from utils.quiver_style import (QUIVER_NEURON_COLORS, QUIVER_COLORS, QUIVER_CYCLE, QUIVER_FONT, QUIVER_RC, apply_quiver_style)
from utils.data_utils  import (FS, load_stim_meta, ap_count_table, load_ensemble_mask_sidecar, stim_mask_from_mat,)
from utils.scs_utils   import (run_scs_pipeline, compute_psp_props_fov, validate_scs_in_part2,
                               PSP_WINDOW)
from utils.sta_utils import (
    run_sta_pipeline, validate_sta_in_part2,        # outdated (kept for reference)
    run_sta_pipeline_interleaved, validate_sta_interleaved,  
    MIN_WIN_DISC, MIN_WIN_VAL, MIN_ONSET_MS,
)
from utils.consensus   import (merge_connections, classify_neurons,
                               neuron_types_from_merged, attach_metadata,)
from utils.visualization import (
    plot_consensus_heatmap,
    plot_top_scs_pair,
    plot_top_sta_pair,
    plot_ap_waveform_sample,
    plot_psp_waveform_sample,
    plot_p1_p2_sta,
    plot_neuron_tier_bar,
    plot_condition_summary,
    plot_validation_rate_by_fov,
    plot_connected_neurons_pct,
    build_event_counts_df,
    plot_event_counts_by_fov,
    plot_validated_conn_count_by_fov,
    plot_spaghetti_overlay,
    plot_full_stacked_traces_batch,
    plot_well_heatmap,
    build_validated_pct_df,
    build_exc_balance_df,
    build_connect_neuron_df,
    build_snr_df,
    plot_dlx_identity_overlay,
    plot_gfp_identity_overlay,
    build_dlx_accuracy_df,
    _find_base_image,
    build_condition_feature_df, 
    plot_condition_heatmap,
    plot_well_feature_heatmap,
    plot_all_sta_connections,
    plot_pharmacology_comparison,
    plot_mean_psp_waveforms,
    plot_pharmacology_summary,
    plot_mean_ap_waveform_by_type,
    classify_dlx_positive, 
    plot_stim_dlx_summary,
    plot_stim_neurons_on_gfp,
    plot_stim_vs_spont_connection_fraction,
    plot_dlx_enrichment_in_inh_connections,
    plot_inh_yield_by_dlx,
)
      
m = re.search(r'PlateNumber_(\d+)', INPUT_DIR);  PLATE_NUMBER = int(m.group(1)) if m else PLATE_NUMBER
m = re.search(r'(QNP-\d+)',          INPUT_DIR);  QNP_ROUND    = m.group(1)     if m else QNP_ROUND
METADATA_PATH = METADATA_PATHS.get(QNP_ROUND, '') if QNP_ROUND else ''
if METADATA_PATH:
    print(f'METADATA_PATH={METADATA_PATH}')
else:
    import warnings
    warnings.warn(
        f"\n{'!'*60}\n"
        f"  NO METADATA FOUND\n"
        f"  Date '{_date}' / QNP_ROUND '{QNP_ROUND}' not in experiment_mapping.csv\n"
        f"  or not in METADATA_PATHS dict.\n"
        f"  Metadata join will be skipped — wellId/drug1 columns will be missing.\n"
        f"  Add the entry to experiment_mapping.csv and METADATA_PATHS before re-running.\n"
        f"{'!'*60}",
        stacklevel=2
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Mode      : {MODE}')
print(f'Input     : {INPUT_DIR}')
print(f'Output    : {OUTPUT_DIR}')
print(f'Round/Plate: {QNP_ROUND}  /  {PLATE_NUMBER}')
print(f'Metadata  : {METADATA_PATH or "(none)"}')
print(f'Pairing window (STA): {PSP_WINDOW} samples = {PSP_WINDOW/FS*1000:.0f} ms, (SCS): 300ms')


# 2. Save analysis parameters for data provenance

In [ ]:
# Save the analysis parameters into the output folder
import json, datetime

_params = {
    'generated':
datetime.datetime.now().isoformat(timespec='seconds'),
    'output_dir':        OUTPUT_DIR,

    # Notebook-level thresholds
    'SCS_SCORE_THRESH':  SCS_SCORE_THRESH,
    'SCS_MIN_AP':        SCS_MIN_AP,
    'STA_SCORE_THRESH':  STA_SCORE_THRESH,
    'STA_MIN_AP':        STA_MIN_AP,

    # SCS pipeline (scs_utils.py)
    'scs_AHP_BLANK_samples':   50,    # 100 ms
    'scs_EPSP_WINDOW_samples': 75,    # 150 ms
    'scs_IPSP_WINDOW_samples': 150,   # 300 ms
    'scs_EPSP_DUR_MIN_ms':     20.0,
    'scs_IPSP_DUR_MIN_ms':     20.0,

    # STA pipeline (sta_utils.py)
    'sta_PSP_WINDOW_samples':  PSP_WINDOW,
    'sta_MIN_ONSET_MS':        MIN_ONSET_MS,
    'sta_MIN_WIN_DISC':        MIN_WIN_DISC,
    'sta_MIN_WIN_VAL':         MIN_WIN_VAL,
    'sta_SNR_AMP_GATE':        4.0,
    'sta_VAL_SNR_RELAXED':     3.5,
    'sta_BASELINE_GATE_SIGMA': 4.0,
    'sta_COINCIDENCE_MS':      5.0,
    'sta_MIN_AP_STIM':         2,

    # AP/trace detection (data_utils.py)
    'FS_hz':             500,
    'SNR_GATE':          3.5,
    'APD50_MAX_samples': 5,
    'AP_RISE_MAX_ms':    20.0,
    'AP_SNR_FRAC':       0.30,
    'PBR_THRESH':        0.6,
}

_params_path = os.path.join(OUTPUT_DIR, 'analysis_parameters.json')
with open(_params_path, 'w') as _f:
    json.dump(_params, _f, indent=2)
print(f'Parameters saved: {_params_path}')

## 3. FOV Discovery

Each FOV may have either spont CSVs, stim CSVs, or both. We discover by the
matching `..._stim_meta.json` (always written) and pick up whichever CSV
pairs are actually present on disk.

In [ ]:
def discover_fov_quartets(input_dir):
    out = []
    metas = sorted(glob.glob(os.path.join(input_dir, '*_stim_meta.json')))
    for meta_path in metas:
        base = meta_path[:-len('_stim_meta.json')]
        sp1 = base + '_spont_part1.csv'
        sp2 = base + '_spont_part2.csv'
        st1 = base + '_stim_part1.csv'
        st2 = base + '_stim_part2.csv'
        m   = re.search(r'(FOV_\d{4})', os.path.basename(base))
        fov = m.group(1) if m else os.path.basename(base)
        has_spont = os.path.exists(sp1) and os.path.exists(sp2)
        has_stim  = os.path.exists(st1) and os.path.exists(st2)
        if not (has_spont or has_stim):
            continue
        out.append(dict(
            fov=fov, meta=meta_path,
            spont_p1=sp1 if has_spont else None,
            spont_p2=sp2 if has_spont else None,
            stim_p1 =st1 if has_stim  else None,
            stim_p2 =st2 if has_stim  else None,
            has_spont=has_spont, has_stim=has_stim,
        ))
    return out

quartets = discover_fov_quartets(INPUT_DIR)
print(f'Discovered {len(quartets)} FOV(s).')
for q in quartets[:2]:
    flags = []
    if q['has_spont']: flags.append('SCS')
    if q['has_stim']:  flags.append('STA')
    print(f'  {q["fov"]:10s}  {"+".join(flags):8s}')
if len(quartets) > 2:
    print(f'  ¦ and {len(quartets) - 2} more')


## 4. Per-FOV Pipeline for resolving quartet of CSV (2 - spontaneous, 2 - stimulation)

Runs whichever branches have data. SCS is skipped on FOVs without
a spont step; STA is skipped on FOVs without a stim step. The consensus
merger handles either being absent.

In [ ]:
# Functions to read the data
def _canonical_excludes(quartet):
    """Union of duplicate-trace IDs across all four CSVs of an FOV."""
    from utils.data_utils import find_duplicate_neurons
    dups = set()
    for key in ('spont_p1','spont_p2','stim_p1','stim_p2'):
        p = quartet.get(key)
        if p and os.path.exists(p):
            dups |= find_duplicate_neurons(pd.read_csv(p))
    return sorted(dups)

# Pipeline runner function
def run_one_fov(quartet, *, scs_thresh=SCS_SCORE_THRESH, sta_thresh=STA_SCORE_THRESH,
                scs_min_ap=SCS_MIN_AP, sta_min_ap=STA_MIN_AP, output_dir=OUTPUT_DIR,
                regress_global=True):
    fov = quartet['fov']
    fov_out = os.path.join(output_dir, fov)
    os.makedirs(fov_out, exist_ok=True)
    meta = load_stim_meta(quartet['meta'])

    excludes = _canonical_excludes(quartet)
    if excludes:
        print(f"  [{fov}] canonical duplicates excluded: {excludes}")

    scs1 = scs2 = None
    scs_df = pd.DataFrame()
    if quartet['has_spont']:
        scs1 = run_scs_pipeline(quartet['spont_p1'], label=f'{fov} spont P1',
                                score_thresh=scs_thresh, output_dir=fov_out,
                                exclude_neurons=excludes, regress_global=regress_global)
        scs2 = run_scs_pipeline(quartet['spont_p2'], label=f'{fov} spont P2',
                                score_thresh=scs_thresh, output_dir=fov_out,
                                exclude_neurons=excludes, regress_global=regress_global)
        scs_df = validate_scs_in_part2(scs1, scs2, score_thresh=scs_thresh,
                                        val_score_thresh=SCS_SCORE_THRESH, min_ap=scs_min_ap)
    else:
        print(f'  [{fov}] no spont CSVs - SCS branch skipped')

    # STA branch - interleaved-trial validation
    sta1 = sta2 = None
    sta_df = pd.DataFrame()
    in_stim_mask_mat = None
    _sidecar = None
    _fov_dir_raw = os.path.join(INPUT_DIR.replace('v17_traces', ''), fov)  # raw FOV dir, above the trace CSVs
    if quartet['has_stim']:
        # Locate EnsembleDmdMasks.mat inside the -1g premovie subfolder
        _mat_matches = glob.glob(os.path.join(_fov_dir_raw, '*-FireflyOne-1g',
                                            'movie1_EnsembleDmdMasks.mat'))
        if _mat_matches:
            try:
                _sidecar = load_ensemble_mask_sidecar(_mat_matches[0])  # loads barcode matrix + mask stack
                in_stim_mask_mat = stim_mask_from_mat(_sidecar, meta)   # {trace_id:bool}, ~50 stim targets
                print(f'  [{fov}] .mat mask: {sum(in_stim_mask_mat.values())} neurons stimulated')
            except Exception as _e:
                print(f'  [{fov}] .mat mask load failed: {_e}')
                in_stim_mask_mat = None
        else:
            print(f'  [{fov}] no EnsembleDmdMasks.mat — falling back to JSON stim_mask_sources')

        sta_res = run_sta_pipeline_interleaved(
            quartet['stim_p1'], quartet['stim_p2'],
            label=f'{fov} stim', score_thresh=sta_thresh, onset_ms=MIN_ONSET_MS,
            output_dir=fov_out, stim_meta=meta,
            in_stim_mask_override=in_stim_mask_mat,  # None → falls back to JSON stim_mask_sources
            exclude_neurons=excludes,
            min_ap_stim=sta_min_ap,
            regress_global=regress_global
        )

        if sta_res is not None:
            from utils.scs_utils import detect_all_morphological_psps
            _eps_morph, _ips_morph = detect_all_morphological_psps(
                sta_res['traces_all'], sta_res['aps_all'], sta_res['segs'],
                raws_all=sta_res.get('raws_all'))
            sta_res['epsps_morphological'] = _eps_morph
            sta_res['ipsps_morphological'] = _ips_morph
            print(f"  [{fov}] Stim morphological PSPs: "
                f"EPSP={sum(len(v) for v in _eps_morph.values())}  "
                f"IPSP={sum(len(v) for v in _ips_morph.values())}")

        sta_df = validate_sta_interleaved(sta_res, score_thresh=sta_thresh)
        sta1 = sta_res
        sta2 = sta_res
    else:
        print(f'  [{fov}] no stim CSVs - STA branch skipped')

    merged = merge_connections(scs_df, sta_df, fov_label=fov)

    all_traces = sorted(set(
        (scs1['all_traces'] if scs1 else []) +
        (sta1['all_traces'] if sta1 else [])
    ))
    scs_nt = neuron_types_from_merged(merged, 'scs_validated')
    sta_nt = neuron_types_from_merged(merged, 'sta_validated')
    neuron_tier = classify_neurons(
        scs_nt, sta_nt, all_traces,
        in_stim_mask=(in_stim_mask_mat if in_stim_mask_mat is not None
                    else (sta1['in_stim_mask'] if sta1 else None)))
    neuron_tier.insert(0, 'fov', fov)

    merged.to_csv(os.path.join(fov_out, f'merged_connections_{fov}.csv'), index=False)
    neuron_tier.to_csv(os.path.join(fov_out, f'merged_neurons_{fov}.csv'), index=False)

    _in_stim_for_dlx = (in_stim_mask_mat if in_stim_mask_mat is not None
                        else (sta1['in_stim_mask'] if sta1 else {}))
    dlx_positive, dlx_stim_summary = {}, None
    if _in_stim_for_dlx and quartet['has_stim']:
        dlx_positive = classify_dlx_positive(
            stim_meta=meta, all_traces=all_traces,
            fov_dir=_fov_dir_raw, fov=fov,
        )
        dlx_stim_summary = plot_stim_dlx_summary(
            in_stim_mask=_in_stim_for_dlx,
            dlx_positive=dlx_positive,
            title=f'{fov} — stimulated neurons: DLX+ vs DLX-',
            output_path=os.path.join(fov_out, f'dlx_stim_summary_{fov}.png'),
        )
    return dict(fov=fov, scs1=scs1, scs2=scs2, sta1=sta1, sta2=sta2,
                scs_df=scs_df, sta_df=sta_df, mat_sidecar=_sidecar,
                merged=merged, neuron_tier=neuron_tier, meta=meta,
                fov_out=fov_out,
                dlx_positive=dlx_positive, dlx_stim_summary=dlx_stim_summary,
                )

In [ ]:
# Run Single FOV (when `MODE='single'`)
if MODE == 'single':
    sel = [q for q in quartets if q['fov'] == SELECTED_FOV]
    if not sel:
        avail = [q['fov'] for q in quartets]
        raise SystemExit(f'{SELECTED_FOV} not found. Available: {avail}')
    t0 = time.time()
    R  = run_one_fov(sel[0])
    print(f'\nSingle FOV done in {time.time()-t0:.1f}s.')
    print(f'  merged connections : {len(R["merged"])}')
    print(f'  per-neuron tiers   : {R["neuron_tier"]["consensus_type"].value_counts().to_dict()}')
else:
    R = None
    print("MODE = 'batch' - skip to section 6 to run all FOVs.")

# QC: EnsembleDmdMasks.mat vs stim_meta.json
def find_ensemble_mat(fov_dir):
      import glob
      matches = glob.glob(os.path.join(fov_dir, '*-FireflyOne-1g', 'movie1_EnsembleDmdMasks.mat'))
      return matches[0] if matches else None

if R is not None:
    fov_dir  = os.path.join(INPUT_DIR.replace('v17_traces', ''), R['fov'])
    mat_path = find_ensemble_mat(fov_dir)

    if mat_path:
        sidecar  = load_ensemble_mask_sidecar(mat_path)
        barcodes = sidecar['barcode_matrix']          # (n_neurons, n_masks)

        stim_mat  = set(np.where(barcodes.any(axis=1))[0] + 1)
        sfps      = R['meta'].get('stim_frames_per_source', {})
        stim_json = set(int(k) for k in sfps)         # keys already T-stripped

        # in_stim_mask from the pipeline (should match stim_json)
        pipeline_mask = R['sta1']['in_stim_mask'] if R['sta1'] else {}
        stim_pipeline = set(t for t, v in pipeline_mask.items() if v)

        print(f"Neurons stimulated - ensemble .mat : {sorted(stim_mat)}")
        print(f"Neurons stimulated - JSON (sfps)   : {sorted(stim_json)}")
        print(f"Neurons stimulated - pipeline mask : {sorted(stim_pipeline)}")
        print(f".mat only  : {sorted(stim_mat - stim_json)}")
        print(f"JSON only  : {sorted(stim_json - stim_mat)}")

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.imshow(barcodes.T, aspect='auto', cmap='Blues', interpolation='none')
        ax.set_xlabel('Neuron index (1-indexed)')
        ax.set_ylabel('Mask pattern')
        ax.set_title(f'{R["fov"]} - EnsembleDmdMasks barcode  '
                    f'(ordering: {sidecar["ordering_of_masks"][:5]}...)')
        plt.tight_layout()
        plt.show()
    else:
        print(f'No EnsembleDmdMasks.mat found for {R["fov"]}')

## 5. Plot waveforms from the top SCS and STA pairs

In [ ]:
## 5. Plot the top scs and sta pairs
def _plot_top_pairs(R):
    if R is None or R['scs1'] is None:
        return
    fov_out = R['fov_out']

    plot_top_scs_pair(R['scs1'], conn_type='exc',
        output_path=os.path.join(fov_out, 'scs_top_exc.png'))
    plot_top_scs_pair(R['scs1'], conn_type='inh',
        output_path=os.path.join(fov_out, 'scs_top_inh.png'))
    plot_top_sta_pair(R['sta1'], conn_type='exc',
        output_path=os.path.join(fov_out, 'sta_top_exc.png'))
    plot_top_sta_pair(R['sta1'], conn_type='inh',
        output_path=os.path.join(fov_out, 'sta_top_inh.png'))
    plot_neuron_tier_bar(R['neuron_tier'],
        output_path=os.path.join(fov_out, 'neuron_tier_bar.png'))


## 6. Batch Pipeline - Run Spont and STA discovery on all available FOVs

In [ ]:
# Load and analyze all of the FOVs in this plate - parallelized
if MODE == 'batch':
    from joblib import Parallel, delayed
    import os as _os

    def _fov_worker(q):
        import matplotlib; matplotlib.use('Agg')
        import gc, traceback
        import matplotlib.pyplot as plt
        fov = q['fov']
        try:
            res = run_one_fov(q)
            return {'ok': True, 'fov': fov, 'res': res}
        except Exception as e:
            return {'ok': False, 'fov': fov, 'error': str(e),
                    'tb': traceback.format_exc()}
        finally:
            plt.close('all'); gc.collect()

    t0 = time.time()
    all_merged, all_neurons, all_fov_results, failed_fovs = [], [], [], []

    n_jobs = min(max(1, _os.cpu_count() - 2), len(quartets))
    print(f"Running {len(quartets)} FOVs on {n_jobs} workers...")

    job_results = Parallel(n_jobs=n_jobs, backend='loky', verbose=5)(
        delayed(_fov_worker)(q) for q in quartets
    )

    for r in job_results:
        fov = r['fov']
        if r['ok']:
            all_merged.append(r['res']['merged'])
            all_neurons.append(r['res']['neuron_tier'].assign(fov=fov))
            all_fov_results.append(r['res'])
            print(f"  [OK] {fov}")
        else:
            print(f"  [FAILED] {fov}: {r['error']}")
            print(r['tb'])
            failed_fovs.append((fov, r['error']))

    df_all_conn    = pd.concat(all_merged,  ignore_index=True) if all_merged  else pd.DataFrame()
    df_all_neurons = pd.concat(all_neurons, ignore_index=True) if all_neurons else pd.DataFrame()

    if not df_all_conn.empty:
        out_conn = os.path.join(OUTPUT_DIR, 'merged_connections_all_fovs.csv')
        df_all_conn.to_csv(out_conn, index=False)
        print(f'\nSaved {out_conn}  ({len(df_all_conn)} rows)')

    if not df_all_neurons.empty:
        out_neur = os.path.join(OUTPUT_DIR, 'merged_neurons_all_fovs.csv')
        df_all_neurons.to_csv(out_neur, index=False)
        print(f'Saved {out_neur}  ({len(df_all_neurons)} rows)')

    print(f'\nBatch complete in {(time.time()-t0)/60:.1f} min - '
        f'{len(quartets)-len(failed_fovs)}/{len(quartets)} succeeded')
    if failed_fovs:
        print(f'Failed: {[f for f,_ in failed_fovs]}')

    fresh_batch_analyzed = True
    import pickle
    cache_path = os.path.join(OUTPUT_DIR, 'all_fov_results_cache.pkl')
    with open(cache_path, 'wb') as f:
        pickle.dump(all_fov_results, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'Saved cache at {cache_path}')
    
    print(f'\nPickle saved {(time.time()-t0)/60:.1f} min - ')
    
    if MODE == 'single': 
        print("MODE != 'batch' - skipping batch loop.")

# ***!!!!!!!!!!6b. Jump in point from previously analyzed batch!!!!!!!!!!!!!!***

In [ ]:
# Reload a previously analyzed dataset
if not LOAD_FROM_CACHE:
    print('LOAD_FROM_CACHE = False — using in-memory batch results.')
else:
    MODE = 'batch'
    import pickle
    cache_path = r'path here\all_fov_results_cache.pkl'
    cache_dir  = os.path.dirname(cache_path)
    OUTPUT_DIR = cache_dir

    if not os.path.exists(cache_path):
        print(f'ERROR: cache not found at {cache_path}')
    else:
        with open(cache_path, 'rb') as f: all_fov_results = pickle.load(f)
        print(f'Loaded {len(all_fov_results)} FOV results from cache: {cache_path}')

    if RELOAD_DATAFRAMES:
        df_all_conn_md = pd.read_csv(os.path.join(cache_dir,'merged_connections_all_fovs_with_metadata.csv'))
        df_all_neurons = pd.read_csv(os.path.join(cache_dir,'merged_neurons_all_fovs.csv'))
        fov_to_well    = df_all_conn_md.groupby('fov')['wellId'].first().to_dict()
        print('Reloaded dataframes from disk.')
        print("WARN: reloaded results only carry 'fov' and 'neuron_tier'. "
            "Cells that touch R['meta']/R['merged']/R['sta1'] will fail.")
    else:
        print('RELOAD_DATAFRAMES = False — keeping in-memory dataframes.')

## 7. Metadata Join + Condition Summary (batch only)

In [ ]:
if MODE == 'batch' and 'df_all_conn' in dir() and not df_all_conn.empty and METADATA_PATH and PLATE_NUMBER:
    keep_cols = ('parametersTested', 'testingCondition', 'drug1', 'wellId')
    df_all_conn_md = attach_metadata(df_all_conn, METADATA_PATH, PLATE_NUMBER,
                                    fov_col='fov', keep_cols=keep_cols)
    out = os.path.join(OUTPUT_DIR, 'merged_connections_all_fovs_with_metadata.csv')
    df_all_conn_md.to_csv(out, index=False)
    print(f'Saved {out}')
    plot_condition_summary(df_all_conn_md, condition_col='drug1',
                            output_path=os.path.join(OUTPUT_DIR, 'condition_summary.png'))
else:
    print('Metadata join skipped (need MODE=batch + valid metadata path + plate number).')
    df_all_conn_md = (df_all_conn.copy()
                    if ('df_all_conn' in dir() and not df_all_conn.empty)
                    else pd.DataFrame())

if 'df_all_conn_md' in dir() and not df_all_conn_md.empty:
    if 'drug1' in df_all_conn_md.columns:
        fov_condition_map = df_all_conn_md.groupby('fov')['drug1'].first().to_dict()
        print('Conditions found:', set(fov_condition_map.values()))
    else:
        fov_condition_map = {}
        print('No drug1 column in metadata — skipping condition map.')
else:
    fov_condition_map = {}
    print('No metadata joined — all FOVs will be No Drug')

In [ ]:
print(f'MODE={MODE}  METADATA_PATH={METADATA_PATH}  PLATE_NUMBER={PLATE_NUMBER}')
print(f'df_all_conn exists: {"df_all_conn" in dir()}  empty: {df_all_conn.empty if "df_all_conn" in dir() else "N/A"}')

# 8. Visualize the data

In [ ]:
import importlib
import utils.visualization
importlib.reload(utils.visualization)
from utils.visualization import (plot_full_stacked_traces_batch, plot_mean_psp_waveforms,
    plot_pharmacology_summary, plot_mean_ap_waveform_by_type, plot_stim_neurons_on_gfp,
    classify_dlx_positive, plot_stim_dlx_summary,plot_stim_vs_spont_connection_fraction,
    plot_dlx_enrichment_in_inh_connections,
    plot_inh_yield_by_dlx,)


In [ ]:
# Plot top connections
# Run for single FOV or every FOV in batch
_results = all_fov_results if MODE == 'batch' else ([R] if R is not None else [])
for _R in _results:
    if _R is not None:
        _plot_top_pairs(_R)

In [ ]:
# Plot stacked traces into a single figure with their stim frames
if MODE == 'batch':
    plot_full_stacked_traces_batch(quartets=quartets, all_fov_results=all_fov_results, output_dir=OUTPUT_DIR);
else:
      plot_full_stacked_traces_batch(quartets=[sel[0]], all_fov_results=[R], output_dir=OUTPUT_DIR)

In [ ]:
# Cell identity accuracy analysis (compare ID'ed presyn neurons to Supplemental figure tags):
if MODE == 'batch':
    df_dlx_fov, df_dlx_summary = build_dlx_accuracy_df(
        all_fov_results, INPUT_DIR, 
        window=5, window_neg=1,        # 1x1 or 3x3 pixel area for DLX-
        thresh_pct=30, thresh_pct_neg=70
        )  # higher percentile = stricter for DLX-
    
    display(df_dlx_summary)
    df_dlx_fov.to_csv(os.path.join(OUTPUT_DIR, 'dlx_accuracy_per_fov.csv'), index=False)
    df_dlx_summary.to_csv(os.path.join(OUTPUT_DIR, 'dlx_accuracy_summary.csv'), index=False)

    _dlx_data = {
      'EXC': df_dlx_fov['pct_exc_correct'].dropna(),
      'INH': df_dlx_fov['pct_inh_correct'].dropna(),
  }

    fig, ax = plt.subplots(figsize=(4, 4))

    for i, (label, vals) in enumerate(_dlx_data.items()):
        n = len(vals)
        mean = vals.mean() if n else 0
        sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
        color = QUIVER_NEURON_COLORS['Inhibitory'] if label == 'INH' else QUIVER_NEURON_COLORS['Excitatory']
        ax.bar(i, mean, yerr=sem, color=color, width=0.5, capsize=6,
                error_kw=dict(elinewidth=1.5), label=f'{label} (n={n} fovs)')
        ax.scatter([i] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
        ax.text(i, 0.5, f'n={n} FOVs', ha='center', va='bottom', fontsize=9)

    #ax.axhline(50, color='gray', linestyle='--', linewidth=1, label='chance')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['EXC','INH'])
    ax.set_ylabel('Cell Identity matches tag identity (%)')
    ax.set_ylim(0, 103)
    ax.set_title('Cell Identity\nDLX- = Excitatory, DLX+ = Inhibitory')
    ax.spines[['top', 'right']].set_visible(False)
    #ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dlx_accuracy_bars.png'), bbox_inches='tight')
    plt.close();

In [ ]:
# Build metrics table for entire plate
if MODE == 'batch':
    import numpy as np
    import pandas as pd
    from IPython.display import display

    _VAL_TIERS = ('both', 'scs_only', 'sta_only')
    _EXC_NT    = ('confident_exc', 'putative_exc')
    _INH_NT    = ('confident_inh', 'putative_inh')
    _SILENT_NT = ('no_valid_outgoing_connections', 'silent')
    # 2x2 binning = one pixel is 13um, accounting for magnification (135mm tube lens / 50mm objective) = 4.81
    _PIXEL_UM  = 4.81 

    def _composite_score(row):
        tier = row['consensus_tier']
        scs  = float(row['scs_score_p1']) if pd.notna(row['scs_score_p1']) else np.nan
        sta  = float(row['sta_score_p1']) if pd.notna(row['sta_score_p1']) else np.nan
        if tier == 'both':
            vals = [v for v in (scs, sta) if not np.isnan(v)]
            return float(np.mean(vals)) if vals else np.nan
        elif tier == 'sta_only':
            return sta if not np.isnan(sta) else scs
        elif tier == 'scs_only':
            return scs if not np.isnan(scs) else sta
        else:  # unvalidated / conflict
            vals = [v for v in (scs, sta) if not np.isnan(v)]
            return float(np.max(vals)) if vals else np.nan

    fov_rows = []
    for R in all_fov_results:
        fov   = R['fov']
        meta  = R['meta']
        mg    = R['merged'].copy()
        nt_df = R.get('neuron_tier')

        mg['cs'] = mg.apply(_composite_score, axis=1)
        val   = mg[mg['consensus_tier'].isin(_VAL_TIERS)]
        unval = mg[~mg['consensus_tier'].isin(_VAL_TIERS)]

        row = {'fov': fov}

        # ### Validated connection counts ##########################################################
        for typ in ('EXC', 'INH'):
            row[f'n_val_{typ.lower()}'] = int((val['type'].str.upper() == typ).sum())

        # ### Composite confidence score (mean per FOV) #####################################
        for subset, lbl in ((val, 'val'), (unval, 'unval')):
            for typ in ('EXC', 'INH'):
                s = subset[subset['type'].str.upper() == typ]['cs'].dropna()
                row[f'score_{lbl}_{typ.lower()}'] = s.mean() if len(s) else np.nan

        # ### Active neurons ##############################################################################
        src = R.get('scs1') or R.get('sta1')
        if src and 'aps_all' in src:
            aps_all = src['aps_all']
            row['n_neurons'] = len(aps_all)
            row['n_active']  = sum(1 for v in aps_all.values() if len(v) > 0)
        elif nt_df is not None:
            row['n_neurons'] = len(nt_df)
            row['n_active']  = int((~nt_df['consensus_type'].isin(_SILENT_NT)).sum())
        else:
            row['n_neurons'] = row['n_active'] = 0

        # ### Cell identity ###############################################################################
        if nt_df is not None and not nt_df.empty:
            n_tot = row['n_neurons']
            n_exc = int(nt_df['consensus_type'].isin(_EXC_NT).sum())
            n_inh = int(nt_df['consensus_type'].isin(_INH_NT).sum())
            row.update(n_id_exc=n_exc, n_id_inh=n_inh,
                        frac_exc=n_exc / n_tot if n_tot else np.nan,
                        frac_inh=n_inh / n_tot if n_tot else np.nan)
        else:
            row.update(n_id_exc=0, n_id_inh=0, frac_exc=np.nan, frac_inh=np.nan)

        # ### Distance (validated connections only, Âµm) #####################################
        c_r = np.asarray(meta.get('source_centroid_row', []), dtype=float)
        c_c = np.asarray(meta.get('source_centroid_col', []), dtype=float)
        for typ in ('EXC', 'INH'):
            dist_val = np.nan
            sub = val[val['type'].str.upper() == typ]
            if c_r.size and not sub.empty:
                pi = sub['pre'].astype(int).values - 1
                qi = sub['post'].astype(int).values - 1
                in_bounds = (pi >= 0) & (pi < c_r.size) & (qi >= 0) & (qi < c_r.size)
                if in_bounds.any():
                    pb, qb = pi[in_bounds], qi[in_bounds]
                    not_nan = ~np.isnan(c_r[pb]) & ~np.isnan(c_r[qb])
                    if not_nan.any():
                        dr = c_r[pb[not_nan]] - c_r[qb[not_nan]]
                        dc = c_c[pb[not_nan]] - c_c[qb[not_nan]]
                        dist_val = float(np.hypot(dr, dc).mean() * _PIXEL_UM)
            row[f'dist_{typ.lower()}_um'] = dist_val

        # ### Hub neurons (>3 validated outgoing connections) ############################
        if not val.empty and nt_df is not None:
            hub_ids = set(val.groupby('pre').size()
                            .pipe(lambda s: s[s > 3]).index.astype(int))
            nt_copy = nt_df.copy()
            nt_copy['trace'] = nt_copy['trace'].astype(int)
            for typ, nt_set in (('exc', _EXC_NT), ('inh', _INH_NT)):
                typed = set(nt_copy[nt_copy['consensus_type'].isin(nt_set)]['trace'])
                row[f'n_hub_{typ}'] = len(hub_ids & typed)
        else:
            row.update(n_hub_exc=0, n_hub_inh=0)

        fov_rows.append(row)

    df_fov = pd.DataFrame(fov_rows)

    # ### Aggregate to plate-level mean / SEM / n ####################################################
    metrics = [
        ('n_val_exc',        'Validated connections',     'EXC'),
        ('n_val_inh',        'Validated connections',     'INH'),
        ('score_val_exc',    'Score - validated',         'EXC'),
        ('score_val_inh',    'Score - validated',         'INH'),
        ('score_unval_exc',  'Score - unvalidated',       'EXC'),
        ('score_unval_inh',  'Score - unvalidated',       'INH'),
        ('n_active',         'Active neurons',            '-'),
        ('n_id_exc',         'Cell identity - count',     'EXC'),
        ('n_id_inh',         'Cell identity - count',     'INH'),
        ('frac_exc',         'Cell identity - fraction',  'EXC'),
        ('frac_inh',         'Cell identity - fraction',  'INH'),
        ('dist_exc_um',      'Distance (µm)',              'EXC'),
        ('dist_inh_um',      'Distance (µm)',              'INH'),
        ('n_hub_exc',        'Hub neurons (>3 val.)',      'EXC'),
        ('n_hub_inh',        'Hub neurons (>3 val.)',      'INH'),
    ]

    summary_rows = []
    for col, metric, cat in metrics:
        s = df_fov[col].dropna()
        n = len(s)
        summary_rows.append({
            'Metric':   metric,
            'Category': cat,
            'Mean':     round(s.mean(), 3) if n else np.nan,
            'SEM':      round(s.std(ddof=1) / np.sqrt(n), 3) if n > 1 else np.nan,
            'n (FOVs)': n,
        })

    df_summary = pd.DataFrame(summary_rows)
    display(df_summary)

    out_path = os.path.join(OUTPUT_DIR, 'plate_summary_table.csv')
    df_summary.to_csv(out_path, index=False)
    print(f'Saved at’ {out_path}')

In [ ]:
# Plot plate metrics per fov
if MODE == 'batch' and not df_all_conn.empty:
    plot_validation_rate_by_fov(
        df_all_conn,
        output_path=os.path.join(OUTPUT_DIR, 'validation_rate_by_fov.png'),
    )
    plot_connected_neurons_pct(
        df_all_conn, df_all_neurons,
        output_path=os.path.join(OUTPUT_DIR, 'connected_neurons_pct.png'),
    )
    
    df_events = build_event_counts_df(all_fov_results)   # list of R dicts
    plot_event_counts_by_fov(
        df_events,
        log_scale=False,
        output_path=os.path.join(OUTPUT_DIR, 'event_counts_per_fov.png'),
    )
    plot_validated_conn_count_by_fov(
          df_all_conn,
          output_path=os.path.join(OUTPUT_DIR, 'validated_conn_count_by_fov.png'),
      )

In [ ]:
# Plot number of action potentials against number of PSPs
if MODE == 'batch' and not df_all_conn.empty:
    from scipy import stats

    fig, ax = plt.subplots(figsize=(5, 5))

    x = df_events['n_aps']
    y = df_events['n_epsps'] + df_events['n_ipsps']

    ax.scatter(x, y, color='steelblue', s=40, alpha=0.7, zorder=5)

    # Correlation line
    slope, intercept, r, p, _ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, color='firebrick', linewidth=1.5,
            label=f'r={r:.2f}, p={p:.3f}')

    ax.set_xlabel('Action potentials per FOV')
    ax.set_ylabel('PSPs per FOV (EPSP + IPSP)')
    ax.set_title('AP vs PSP counts per FOV')
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    for ext in ['svg', 'png']:
        plt.savefig(os.path.join(OUTPUT_DIR, f'ap_vs_psp_correlation.{ext}'), bbox_inches='tight')
    plt.close()

In [ ]:
# Plot the confidence scores for E and I connections
if MODE == 'batch':
    _score_data = {
        'EXC': df_fov['score_val_exc'].dropna(),
        'INH': df_fov['score_val_inh'].dropna(),
    }

    fig, ax = plt.subplots(figsize=(4, 4))

    for i, (label, vals) in enumerate(_score_data.items()):
        n = len(vals)
        mean = vals.mean() if n else 0
        sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
        color = QUIVER_NEURON_COLORS['Inhibitory'] if label == 'INH' else QUIVER_NEURON_COLORS['Excitatory']
        ax.bar(i, mean, yerr=sem, color=color, width=0.5, capsize=6,
                error_kw=dict(elinewidth=1.5), label=f'{label} (n={n} FOVs)')
        ax.scatter([i] * n, vals, color='k', s=20, zorder=5, alpha=0.6)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['EXC', 'INH'])
    ax.set_ylabel('Confidence score (validated connections)')
    ax.set_title('Validated connection confidence')
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    for ext in ['svg', 'png']:
        plt.savefig(os.path.join(OUTPUT_DIR, f'confidence_score_bars.{ext}'), bbox_inches='tight')
    plt.close()

In [ ]:
# Plot mean PSP waveforms (average across all connected pairs)
if MODE == 'batch':
    for R in all_fov_results:
        fov = R.get('fov', 'unknown')
        plot_mean_psp_waveforms(
            R,
            output_path=os.path.join(OUTPUT_DIR, fov, f'{fov}_mean_psp_waveforms.png')
        )

In [ ]:
# Pie charts for each FOV of E and I and no connections
if MODE == 'batch':
    DISPLAY_ORDER = ['Excitatory', 'Inhibitory', 'Ambiguous',
                     'No validated efferent connections']
    colors = ['red', 'lightskyblue', 'yellowgreen', 'gray']

    for R in all_fov_results:
        counts = (R['neuron_tier']['consensus_type']
                  .map(DISPLAY_NAMES)
                  .value_counts()
                  .reindex(DISPLAY_ORDER, fill_value=0))

        mask  = counts.values > 0
        sizes = counts.values[mask]
        lbls  = [l for l, m in zip(DISPLAY_ORDER, mask) if m]
        clrs  = [c for c, m in zip(colors, mask) if m]

        fig, ax = plt.subplots()
        patches, texts, autotexts = ax.pie(
            sizes, labels=lbls, autopct='%1.1f%%',
            colors=clrs, startangle=140)
        for text, color in zip(texts, clrs):
            text.set_color(color)
            text.set_fontweight('bold')
        ax.set_title(R['fov'])
        ax.axis('equal')

        fov_dir = os.path.join(OUTPUT_DIR, R['fov'])
        os.makedirs(fov_dir, exist_ok=True)
        for ext in ['svg', 'png']:
            plt.savefig(os.path.join(fov_dir, f'pie_chart.{ext}'), bbox_inches='tight')
        plt.close()

In [ ]:
# # Pie chart: consensus neuron type distribution across all FOVs (batch)
if MODE == 'batch':
    DISPLAY_ORDER = ['Excitatory', 'Inhibitory', 'Ambiguous',
                     'No validated efferent connections']
    colors = [
        "#9C02FA",   # _EXC  — Quiver purple
        "#F51D3F",   # _INH  — Quiver red
        "#FFFFFF",   # _WHITE — ambiguous
        "#888888",   # _GREY — no connections
    ]

    counts = (df_all_neurons['consensus_type']
              .map(DISPLAY_NAMES)
              .value_counts()
              .reindex(DISPLAY_ORDER, fill_value=0))

    mask  = counts.values > 0
    sizes = counts.values[mask]
    lbls  = [l for l, m in zip(DISPLAY_ORDER, mask) if m]
    clrs  = [c for c, m in zip(colors, mask) if m]

    fig, ax = plt.subplots()
    patches, texts, autotexts = ax.pie(
        sizes, labels=lbls, autopct='%1.1f%%',
        colors=clrs, startangle=140)
    for text, color in zip(texts, clrs):
        text.set_color(color)
        text.set_fontweight('bold')
    ax.set_title(f'All FOVs pooled (n={len(df_all_neurons)} neurons)')
    ax.axis('equal')
    plt.savefig(os.path.join(OUTPUT_DIR, 'pie_chart_all_fovs.png'), bbox_inches='tight')
    plt.close()
    print(f'batch mode selected')

In [ ]:
# Spaghetti plot generation
if MODE == 'batch':
    plt.ioff()
    for R in all_fov_results:
        fov_dir = os.path.join(
        INPUT_DIR.replace('v17_traces', ''),
        R['fov'])
        aps = (R['scs1']['aps_all'] if R['scs1'] else
            R['sta1']['aps_all'] if R['sta1'] else None)
        plot_spaghetti_overlay(
            R['merged'],
            R['neuron_tier'],
            R['meta'],
            aps_all=aps,
            score_thresh=STA_SCORE_THRESH,
            fov_dir=fov_dir, channel='quasr',
            output_path=os.path.join(R['fov_out'], 'spaghetti_quasr.png'))
        plt.close()
    plt.ion()
    print(f'batch mode selected')

In [ ]:
# Well-level heatmaps (1 FOV per well — FOV-level plots are redundant)
if MODE == 'batch':
    _has_live_results = bool(all_fov_results) and 'scs1' in all_fov_results[0]
    if not _has_live_results:
        print("WARNING: all_fov_results is from a CSV reload — "
              "plots requiring live batch data (active neurons, SNR) "
              "will be skipped or show proxy values. Re-run the batch for full results.")

    fov_to_well = df_all_conn_md.groupby('fov')['wellId'].first().to_dict()
    fov_to_drug = df_all_conn_md.groupby('fov')['drug1'].first().to_dict() \
                  if 'drug1' in df_all_conn_md.columns else {}

    df_val = build_validated_pct_df(df_all_conn_md)
    df_snr = build_snr_df(all_fov_results, fov_to_well, snr_gate=3.5, fov_to_drug=fov_to_drug)
    df_exc = build_exc_balance_df(df_all_conn_md)
    df_act = build_connect_neuron_df(all_fov_results, fov_to_well, fov_to_drug=fov_to_drug)

    # 1. % validated connections
    plot_well_heatmap(df_val, 'pct_validated', '% validated connections per well',
        output_path=os.path.join(OUTPUT_DIR, 'well_heatmap_validated.png'),
        cmap='Purples', cbar_label='% validated')

    # 2. Exc/Inh balance
    plot_well_heatmap(df_exc, 'exc_balance', 'EXC/INH balance per well (# validated connections)',
        output_path=os.path.join(OUTPUT_DIR, 'well_heatmap_exc_balance.png'),
        cmap='RdYlGn', vmin=-1, vmax=1, fmt='.2f',
        cbar_label='all INH  |  all EXC')

    # 3. % neurons with 1+ connection
    if not _has_live_results:
        print("  [active neurons] using neuron tier proxy — re-run batch for AP-based counts")
    plot_well_heatmap(df_act, 'pct_connected', '% active neurons with connections per well',
        output_path=os.path.join(OUTPUT_DIR, 'well_heatmap_connected_neurons.png'),
        cmap='Purples', cbar_label='% connected neurons')

    # 4. % neurons passing SNR gate
    if not _has_live_results:
        print("  [SNR] skipped — requires live batch results")
    else:
        plot_well_heatmap(df_snr, 'pct_above_snr', '% neurons mean SNR >= 3.5 per well',
            output_path=os.path.join(OUTPUT_DIR, 'well_heatmap_snr.png'),
            cmap='Purples', cbar_label='% neurons above SNR gate')

In [ ]:
# Plot AP waveforms by Exc or Inh
if MODE == 'batch':
    plot_mean_ap_waveform_by_type(
        all_fov_results,
        output_path=os.path.join(OUTPUT_DIR, 'mean_ap_waveform_by_type.png')
    )

In [ ]:
# Plot the distance of validated connections
if MODE == 'batch':
    _dist_data = {
        'EXC': df_fov['dist_exc_um'].dropna(),
        'INH': df_fov['dist_inh_um'].dropna(),
    }
    fig, ax = plt.subplots(figsize=(4, 4))
    for i, (label, vals) in enumerate(_dist_data.items()):
        n = len(vals)
        mean = vals.mean() if n else 0
        sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
        color = QUIVER_NEURON_COLORS['Inhibitory'] if label == 'INH' else QUIVER_NEURON_COLORS['Excitatory']
        ax.bar(i, mean, yerr=sem, color=color, width=0.5, capsize=6,
                error_kw=dict(elinewidth=1.5), label=f'{label} (n={n} fovs)')
        ax.scatter([i] * n, vals, color='k', s=20, zorder=5, alpha=0.6)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['EXC', 'INH'])
    ax.set_ylabel('Mean connection distance (µm)')
    ax.set_title('Connection distance by neuron type')
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dist_exc_inh.png'), bbox_inches='tight')
    plt.close()

In [ ]:
# STA connection grids — plot all the validated pairs from STA, saved per FOV
if MODE == 'batch':
    for R in all_fov_results:
        if R.get('sta1') is None:
            continue
        fov_dir = os.path.join(OUTPUT_DIR, R['fov'])
        os.makedirs(fov_dir, exist_ok=True)
        plot_all_sta_connections(R, 'exc', output_path=os.path.join(fov_dir, 'sta_all_exc.png'))
        plot_all_sta_connections(R, 'inh', output_path=os.path.join(fov_dir, 'sta_all_inh.png'));

In [ ]:
def _plot_dlx_protocol_figures(R):
    if R is None:
        return
    fov     = R['fov']
    fov_out = R['fov_out']

    plot_stim_vs_spont_connection_fraction(R,
        output_path=os.path.join(fov_out, f'dlx_conn_fraction_{fov}.png'))
    plot_dlx_enrichment_in_inh_connections(R,
        output_path=os.path.join(fov_out, f'dlx_enrichment_inh_{fov}.png'))
    plot_inh_yield_by_dlx(R,
        output_path=os.path.join(fov_out, f'dlx_inh_yield_{fov}.png'))

# Run for single FOV or every FOV in batch
_results = all_fov_results if MODE == 'batch' else ([R] if R is not None else [])
for _R in _results:
    if _R is not None:
        _plot_dlx_protocol_figures(_R)

# 9. Quality control figures - batch analysis

In [ ]:
# Cell Identity: generate DLX and GFP tag overlay plots
if MODE == 'batch':
    for R in all_fov_results:
        fov_dir = os.path.join(INPUT_DIR.replace('v17_traces', ''), R['fov'])
        plot_dlx_identity_overlay(
            R, fov_dir,
            output_path=os.path.join(R['fov_out'], f"SupplementalImages_RCaMP_{R['fov']}.png"))
        plot_gfp_identity_overlay(
            R, fov_dir,
            output_path=os.path.join(R['fov_out'], f"SupplementalImages_GFP_{R['fov']}.png"))
    print('DLX and GFP overlay plots saved.')

In [ ]:
# Cell identity accuracy analysis:
if MODE == 'batch':
    df_dlx_fov, df_dlx_summary = df_dlx_fov, df_dlx_summary = build_dlx_accuracy_df(
    all_fov_results, INPUT_DIR, window=5, window_neg=1,        # 1x1 or 3x3 pixel area for DLX-
    thresh_pct=30, thresh_pct_neg=70  # higher percentile = stricter for DLX-
  )
    display(df_dlx_summary)
    df_dlx_fov.to_csv(os.path.join(OUTPUT_DIR, 'dlx_accuracy_per_fov.csv'), index=False)
    df_dlx_summary.to_csv(os.path.join(OUTPUT_DIR, 'dlx_accuracy_summary.csv'), index=False)
    print(f'Saved at {OUTPUT_DIR}')


    _dlx_data = {
      'EXC': df_dlx_fov['pct_exc_correct'].dropna(),
      'INH': df_dlx_fov['pct_inh_correct'].dropna(),
  }

    fig, ax = plt.subplots(figsize=(4, 4))

    for i, (label, vals) in enumerate(_dlx_data.items()):
        n = len(vals)
        mean = vals.mean() if n else 0
        sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
        color = QUIVER_NEURON_COLORS['Inhibitory'] if label == 'INH' else QUIVER_NEURON_COLORS['Excitatory']
        ax.bar(i, mean, yerr=sem, color=color, width=0.5, capsize=6,
                error_kw=dict(elinewidth=1.5), label=f'{label} (n={n} fovs)')
        ax.scatter([i] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
        ax.text(i, 0.5, f'n={n} FOVs', ha='center', va='bottom', fontsize=9)

    #ax.axhline(50, color='gray', linestyle='--', linewidth=1, label='chance')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['EXC','INH'])
    ax.set_ylabel('Cell Identity matches tag identity (%)')
    ax.set_ylim(0, 103)
    ax.set_title('Cell Identity\nDLX- = Excitatory, DLX+ = Inhibitory')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'dlx_accuracy_bars.png'), bbox_inches='tight')
    plt.close()

In [ ]:
# DLX inhibitory classification for stimulated neurons
if MODE == 'batch':
  df_dlx = pd.DataFrame([
        {'fov': res['fov'], **res['dlx_stim_summary']}
        for res in all_fov_results
        if res.get('dlx_stim_summary') is not None
    ])
  print(df_dlx)
  print(df_dlx[['dlx_pos','dlx_neg']].sum())
  print(f"Plate total: {df_dlx['dlx_pos'].sum()} inhibitory / {df_dlx['stim_total'].sum()}stimulated")

In [ ]:
# Compute the PSP properties
if MODE == 'batch':
    psp_rows = []
    for R in all_fov_results:
        props = compute_psp_props_fov(R)
        if props is None:
            continue
        row = {'fov': R['fov']}
        for typ in ('EXC', 'INH'):
            for k, v in props[typ].items():
                row[f'{k}_{typ.lower()}'] = v
        psp_rows.append(row)

    df_psp = pd.DataFrame(psp_rows)


    _psp_props = [
            ('amplitude',      'Amplitude (ΔF/F)'),
            ('auc',            'AUC (ΔF/F · s)'),
            ('peak_time_ms',   'Peak time (ms)'),
            ('onset_delay_ms', 'Onset delay (ms)'),
            ('rise_time_ms',   'Rise time (ms)'),
            ('half_width_ms',  'Half-width (ms)'),
            ('snr',            'SNR'),
            ('decay_time_ms',  'Decay time to 50% (ms)'),
            ('duration_ms',    'Duration above threshold (ms)'),
        ]

    fig, axes = plt.subplots(3, 3, figsize=(12, 10))
    for ax, (prop, ylabel) in zip(axes.flatten(), _psp_props):
        for i, (label, color) in enumerate([('EXC', '#9C02FA'), ('INH', '#F51D3F')]):
            vals = df_psp[f'{prop}_{label.lower()}'].dropna()
            n    = len(vals)
            mean = vals.mean() if n else 0
            sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
            ax.bar(i, mean, yerr=sem, color=color, width=0.5, capsize=6,
                    error_kw=dict(elinewidth=1.5), label=f'{label} (n={n})')
            ax.scatter([i] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['EXC', 'INH'])
        ax.set_ylabel(ylabel)
        ax.legend(frameon=False, fontsize=8)
        ax.spines[['top', 'right']].set_visible(False)

    plt.suptitle('PSP properties — validated connections')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'psp_properties.png'),
    bbox_inches='tight')
    plt.close()

In [ ]:
# Well features heatmap
if MODE == 'batch':
    plt.ioff()
    df_features = build_condition_feature_df(all_fov_results, df_dlx_fov=df_dlx_fov)

    if fov_condition_map:
        df_features['condition'] = df_features['fov'].map(fov_condition_map).fillna('No Drug')

    df_features['plate'] = PLATE_NUMBER

    if 'df_all_conn_md' in dir() and not df_all_conn_md.empty and 'wellId' in df_all_conn_md.columns:
        fov_to_well = df_all_conn_md.groupby('fov')['wellId'].first().to_dict()
        df_features['well'] = df_features['fov'].map(fov_to_well)
    else:
        df_features['well'] = None
        print('No well info available — skipping well assignment')

    plot_well_feature_heatmap(df_features,
                              output_path=os.path.join(OUTPUT_DIR, 'well_feature_heatmap.png'))
    plt.ion()

In [ ]:
# Compare connections in the presence of inhibitory or excitatory synaptic blockers
# Skipped automatically if no relevant wells are found in fov_condition_map.
if MODE == 'batch':
    df_cmp_inh = plot_pharmacology_comparison(
        all_fov_results,
        fov_condition_map,
        conn_type='inh',
        drug_keyword='GABAzine',
        output_path=os.path.join(OUTPUT_DIR, 'nodrug_vs_gabazine_inh.png'),
    )
    # Compare No Drug to glutamate blockers
    df_cmp_exc = plot_pharmacology_comparison(
        all_fov_results,
        fov_condition_map,
        conn_type='exc',
        drug_keyword='AP5-NBQX-CNQX', 
        output_path=os.path.join(OUTPUT_DIR, 'nodrug_vs_apv_exc.png'),
    )

In [ ]:
# Cell — Pharmacology summary figure
if MODE == 'batch':
    plot_pharmacology_summary(df_features, all_fov_results=all_fov_results, 
                            output_path=os.path.join(OUTPUT_DIR, 'pharmacology_summary.png')) 

# Quality control tests for pipeline

In [ ]:
# QC figure for stimulation raster
def plot_stim_raster(res, part='both'):
    """
    Raster plot of stimulation frames per neuron.

    REQUIRES the .mat sidecar — JSON `stim_frames_per_source` is intentionally
    NOT used as a fallback because it carries phantom deliveries from the
    initial 2-stim pair-generation phase that get overridden in the real
    protocol.

    Strict alignment check: ordering_of_masks must have exactly one entry per
    event in [stim_frames_part1; stim_frames_part2]. If lengths disagree, we
    cannot say which frame goes with which mask, so we skip the raster rather
    than draw a misleading one.
    """
    meta  = res['meta']
    split = meta.get('stim_split_local', 0)
    total_frames = meta['stim_n_samples']

    all_traces = res['sta1']['all_traces']
    n_neurons  = len(all_traces)
    tid_to_row = {tid: i for i, tid in enumerate(all_traces)}

    sidecar = res.get('mat_sidecar')
    if sidecar is None:
        print(f"  [{res['fov']}] no mat_sidecar — JSON frames are contaminated "
              f"with overridden initial-pair stims, so a raster cannot be built. "
              f"Skipping.")
        return

    ordering  = np.asarray(sidecar['ordering_of_masks'], dtype=int).flatten()
    barcode   = np.asarray(sidecar['barcode_matrix'], dtype=bool)
    p1_frames = np.asarray(meta.get('stim_frames_part1', []), dtype=float)
    p2_frames = np.asarray(meta.get('stim_frames_part2', []), dtype=float)
    n_uniq    = barcode.shape[1]
    n_p1, n_p2 = len(p1_frames), len(p2_frames)
    total_json_events = n_p1 + n_p2

    need_p1 = part in ('part1', 'both')
    need_p2 = part in ('part2', 'both')

    if len(ordering) != total_json_events:
        print(f"  [{res['fov']}] WARN: ordering_of_masks length ({len(ordering)}) "
              f"!= stim_frames_part1+part2 ({n_p1}+{n_p2}={total_json_events}). "
              f"Cannot align mask-per-event with frame-per-event. Skipping.")
        print(f"             (n_unique_masks={n_uniq}, barcode_neurons={barcode.shape[0]})")
        return

    rows_col, frames_col = [], []
    for i, mask_col in enumerate(ordering):
        col = int(mask_col) - 1
        if col < 0 or col >= n_uniq:
            continue
        # i indexes into the virtual [p1_frames; p2_frames] concatenation.
        if i < n_p1:
            if not need_p1:
                continue
            frame_global = float(p1_frames[i])
        else:
            if not need_p2:
                continue
            frame_global = float(p2_frames[i - n_p1]) + split
        hit_neurons = np.where(barcode[:, col])[0] + 1   # 1-indexed
        for tid in hit_neurons:
            if tid in tid_to_row:
                rows_col.append(tid_to_row[tid])
                frames_col.append(frame_global)

    rows_col   = np.array(rows_col,   dtype=float)
    frames_col = np.array(frames_col, dtype=float)
    src_label  = '.mat barcode'

    # Plot
    fov = res['fov']
    fig, ax = plt.subplots(figsize=(12, 6))
    if len(frames_col):
        ax.scatter(frames_col, rows_col, s=2, c='royalblue',
                   linewidths=0, rasterized=True)
    else:
        ax.text(0.5, 0.5, 'No stim frames found',
                transform=ax.transAxes, ha='center')

    n_stim = len(np.unique(rows_col)) if len(rows_col) else 0
    ax.set_xlim(0, total_frames)
    ax.set_ylim(-0.5, n_neurons - 0.5)
    ax.invert_yaxis()
    ax.set_xlabel('Frame')
    ax.set_ylabel('Neuron index')
    ax.set_title(f'{fov} - stimulation raster  ({n_neurons} neurons, '
                 f'{n_stim} stimulated)  [{src_label}]')
    ax.set_yticks(np.arange(0, n_neurons, max(1, n_neurons // 20)))
    plt.tight_layout()
    out_path = os.path.join(res['fov_out'], f'stim_raster_{fov}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {out_path}  [{src_label}]')

In [ ]:
# Plotting function for quasar images across wells.  
from matplotlib.patches import Rectangle
def plot_well_quasr(all_fov_results, *, fov_to_well=None,
                    n_plate_rows=8, n_plate_cols=12,
                    fov_subrows=None,
                    preference='quasr',
                    output_path=None):
    """
    Plate-layout view with quasr/DLX thumbnail images in each well cell.
    Multiple FOVs per well are stacked vertically as sub-rows.

    Parameters
    ----------
    all_fov_results : list of dicts from run_one_fov()
    n_plate_rows    : int, plate rows (default 8 for 96-well)
    n_plate_cols    : int, plate cols (default 12 for 96-well)
    fov_subrows     : int or None - if None, inferred from max FOVs per well
    preference      : 'quasr' or 'dlx'
    output_path     : str or None
    """
    _EMPTY = '#e8e8e8'

    # ### build per-well list of (fov_label, image_path) ############################
    # Assumes FOV name encodes well, e.g. "B03_FOV1" or "B03"
    from collections import defaultdict
    well_fovs = defaultdict(list)   # (row_idx, col_idx) -> [(label, img_path), ...]

    for R in all_fov_results:
        fov      = R['fov']
        meta     = R['meta']
        savefast = meta.get('savefast', '')
        fov_dir  = os.path.join(
            INPUT_DIR.replace('v17_traces', ''),
            fov)
        img_path = _find_base_image(savefast, fov_dir, preference=preference)

        # ### parse well position from fov_to_well lookup ###########################
        well = fov_to_well.get(fov) if fov_to_well else None
        if not well:
            print(f'  [warn] no well mapping for fov: {fov}')
            continue
        m = re.match(r'^([A-Ha-h])(\d{1,2})$', str(well).strip())
        if not m:
            print(f'  [warn] cannot parse well id: {well}')
            continue
        row_idx = ord(m.group(1).upper()) - ord('A')
        col_idx = int(m.group(2)) - 1
        fov_label = fov.split('_')[-1] if '_' in fov else fov
        well_fovs[(row_idx, col_idx)].append((fov_label, img_path))

    # ### infer fov_subrows if not supplied ################################################
    if fov_subrows is None:
        fov_subrows = max((len(v) for v in well_fovs.values()), default=1)

    # ### figure setup ###############################################################################
    cell_w, cell_h = 1.8, 1.8
    fig, ax = plt.subplots(figsize=(n_plate_cols * cell_w,
                                    n_plate_rows * cell_h))
    ax.set_xlim(0, n_plate_cols)
    ax.set_ylim(n_plate_rows, 0)
    ax.set_aspect('equal')
    ax.axis('off')

    subrow_h = 1.0 / fov_subrows   # fractional height of each sub-row

    # ### empty background sub-rows #############################################################
    for r in range(n_plate_rows):
        for c in range(n_plate_cols):
            for s in range(fov_subrows):
                ax.add_patch(Rectangle(
                    (c, r + s * subrow_h), 1, subrow_h,
                    linewidth=0.5, edgecolor='#aaaaaa',
                    facecolor=_EMPTY, zorder=1))

    # ### fill wells with thumbnails ############################################################
    for (row_idx, col_idx), fov_list in well_fovs.items():
        if row_idx >= n_plate_rows or col_idx >= n_plate_cols:
            continue
        for s, (label, img_path) in enumerate(fov_list):
            if s >= fov_subrows:
                break

            # sub-row bounding box in axis coordinates
            x0 = col_idx
            y0 = row_idx + s * subrow_h
            w  = 1.0
            h  = subrow_h

            if img_path and os.path.isfile(img_path):
                try:
                    img = plt.imread(img_path)
                    # Convert axis coords -> figure coords for inset
                    ax_bbox   = ax.get_position()         # in figure fraction
                    fig_w_in  = fig.get_figwidth()
                    fig_h_in  = fig.get_figheight()

                    # Place image using inset_axes driven by data coords
                    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
                    # We use ax.transData to place an inset at exact data coords
                    from matplotlib.transforms import Bbox, TransformedBbox
                    axins = ax.inset_axes(
                        [x0, y0, w, h],
                        transform=ax.transData,
                        zorder=2)
                    axins.imshow(img, aspect='auto', interpolation='bilinear', cmap='gray')
                    axins.axis('off')
                    axins.text(0.5, 0.02, label,
                               transform=axins.transAxes,
                               ha='center', va='bottom',
                               fontsize=5, color='white',
                               bbox=dict(facecolor='black',
                                         alpha=0.4, pad=1, linewidth=0))
                except Exception as e:
                    print(f'  [warn] could not load image {img_path}: {e}')
                    _draw_empty_subrow(ax, x0, y0, w, h, label, _EMPTY)
            else:
                _draw_empty_subrow(ax, x0, y0, w, h, label, _EMPTY)

    # ### well outlines on top #####################################################################
    for r in range(n_plate_rows):
        for c in range(n_plate_cols):
            ax.add_patch(Rectangle((c, r), 1, 1,
                         linewidth=1.5, edgecolor='#333333',
                         facecolor='none', zorder=3))

    # ### row/col labels ##############################################################################
    for r in range(n_plate_rows):
        ax.text(-0.15, r + 0.5, chr(ord('A') + r),
                ha='right', va='center', fontsize=8, fontweight='bold')
    for c in range(n_plate_cols):
         ax.text(c + 0.5, -0.15, str(c + 1), ha='center', va='bottom', fontsize=8, fontweight='bold'),

    title_pref = 'Quasr (Red)' if preference == 'quasr' else 'DLX (Yellow)'
    ax.set_title(f'Plate layout - {title_pref} thumbnails',
                 fontsize=11, pad=10)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f'Saved {output_path}')
    plt.close()

def _draw_empty_subrow(ax, x0, y0, w, h, label, facecolor):
    """Fallback: grey sub-row with text label when no image is available."""
    ax.add_patch(Rectangle((x0, y0), w, h,
                 linewidth=0.5, edgecolor='#aaaaaa',
                 facecolor=facecolor, zorder=2))
    ax.text(x0 + 0.5, y0 + h / 2, label,
            ha='center', va='center', fontsize=5, color='#666666')

In [ ]:
# Plot all of the quasr images in their real locations to run quality control
# on any light path effects
#if MODE == 'batch':
 #   plot_well_quasr(all_fov_results, fov_to_well=fov_to_well, 
  #                  preference='quasr', output_path=os.path.join(OUTPUT_DIR, 'plate_quasr_thumbnails.png'))

In [ ]:
"""
checks.py — alignment / indexing sanity checks for pipeline v17.

Usage in the notebook:
    csv_paths = {
        "spont1": q.get("spont_p1"),
        "spont2": q.get("spont_p2"),
        "stim1":  q.get("stim_p1"),
        "stim2":  q.get("stim_p2"),
    }
    report = assert_pipeline_consistency(R, csv_paths=csv_paths)
    display(report)
"""

import numpy as np
import pandas as pd


def assert_pipeline_consistency(R, csv_paths=None, verbose=True, strict=False):
    """
    Run CP-1 .. CP-10 on a single FOV's result dict.

    Parameters
    ----------
    R : dict
        Per-FOV result. Recognised keys (all optional):
            fov, meta, scs1, scs2, sta1, sta2, merged, neuron_tier.
    csv_paths : dict | None
        Optional {'spont1','spont2','stim1','stim2'} → file paths. Enables
        CSV-vs-JSON column check (CP-1) and the round-trip check (CP-9).
    verbose : bool
        Print pass/fail/warn for each check.
    strict : bool
        If True, raise AssertionError when any check reports FAIL.

    Returns
    -------
    pd.DataFrame with columns [check, status, detail].
    """
    rows = []

    def log(check, status, detail=""):
        rows.append({"check": check, "status": status, "detail": detail})
        if verbose:
            tag = {"PASS": "✓", "WARN": "!", "FAIL": "✗", "SKIP": "·"}.get(status, "?")
            print(f"  [{tag} {status}] {check:<14}  {detail}")

    fov  = R.get("fov", "(unknown FOV)")
    meta = R.get("meta") or {}
    if verbose:
        print(f"\n=== Pipeline consistency: {fov} ===")

    n_sources = int(meta.get("n_sources", 0) or 0)
    scs = R.get("scs1") or R.get("scs")
    sta = R.get("sta1") or R.get("sta")

    # ── CP-1: CSV column names are T1..TN and match meta.n_sources ──────────
    if csv_paths:
        for label, path in csv_paths.items():
            if not path:
                continue
            try:
                cols = list(pd.read_csv(path, nrows=0).columns)
                parsed = [int(c[1:]) for c in cols if c and c.startswith("T")]
                if parsed != list(range(1, len(cols) + 1)):
                    first_bad = next(
                        (i for i, (a, b) in enumerate(zip(parsed, range(1, len(cols)+1))) if a != b),
                        None,
                    )
                    log(f"CP-1[{label}]", "FAIL",
                        f"column IDs not 1..N (first mismatch at col {first_bad}: "
                        f"got T{parsed[first_bad] if first_bad is not None else '?'})")
                elif n_sources and len(cols) != n_sources:
                    log(f"CP-1[{label}]", "FAIL",
                        f"{len(cols)} columns vs meta.n_sources={n_sources}")
                else:
                    log(f"CP-1[{label}]", "PASS",
                        f"{len(cols)} columns T1..T{len(cols)}")
            except Exception as e:
                log(f"CP-1[{label}]", "FAIL", f"{type(e).__name__}: {e}")
    else:
        log("CP-1", "SKIP", "no csv_paths provided")

    # ── CP-2: SCS and STA agree on the duplicate-excluded all_traces set ────
    if scs and sta:
        s_set = set(int(t) for t in scs.get("all_traces", []))
        a_set = set(int(t) for t in sta.get("all_traces", []))
        only_s, only_a = sorted(s_set - a_set), sorted(a_set - s_set)
        if only_s or only_a:
            log("CP-2", "WARN",
                f"SCS-only={only_s}  STA-only={only_a}  "
                f"(find_duplicate_neurons gave different sets — see review note 2a)")
        else:
            log("CP-2", "PASS", f"both branches see {len(s_set)} neurons")
    else:
        log("CP-2", "SKIP", "need both scs and sta results")

    # ── CP-3: in_stim_mask agrees with stim_frames_per_source non-emptiness ─
    mask = set(int(x) for x in meta.get("stim_mask_sources", []))
    sfps = meta.get("stim_frames_per_source", {}) or {}
    has_frames = set()
    for k, parts in sfps.items():
        if not isinstance(parts, dict):
            continue
        try:
            tid = int(str(k)[1:]) if str(k).startswith("T") else int(k)
        except ValueError:
            continue
        _p1 = parts.get("part1"); p1 = np.asarray(_p1 if _p1 is not None else [], dtype=float)
        _p2 = parts.get("part2"); p2 = np.asarray(_p2 if _p2 is not None else [], dtype=float)
        if len(p1) or len(p2):
            has_frames.add(tid)
    only_mask  = sorted(mask - has_frames)
    only_frame = sorted(has_frames - mask)
    if only_frame:
        log("CP-3", "FAIL",
            f"sources with stim frames but NOT in stim_mask_sources:{only_frame}")
    elif only_mask:
        log("CP-3", "WARN",
            f"in stim mask but no per-source frames: {only_mask} "
            f"(unstimmed mask members — confirm with stim raster)")
    else:
        log("CP-3", "PASS", f"{len(mask)} mask sources, all with stim frames")

    # ── CP-4: centroid arrays length matches n_sources ──────────────────────
    rr = np.asarray(meta.get("source_centroid_row", []), dtype=float)
    cc = np.asarray(meta.get("source_centroid_col", []), dtype=float)
    if n_sources == 0:
        log("CP-4", "SKIP", "n_sources missing from meta")
    elif len(rr) != n_sources or len(cc) != n_sources:
        log("CP-4", "FAIL",
            f"centroid lengths rows={len(rr)} cols={len(cc)} vs n_sources={n_sources}")
    else:
        nan_ids = (np.where(np.isnan(rr) | np.isnan(cc))[0] + 1).tolist()  # 1-indexed
        log("CP-4", "PASS",
            f"len={n_sources}; missing centroid for {len(nan_ids)} sources"
            + (f": {nan_ids}" if 0 < len(nan_ids) <= 10 else ""))

    # ── CP-5: detected APs are inside detected segments ─────────────────────
    for label, res in [("scs", scs), ("sta", sta)]:
        if not res:
            continue
        aps_all = res.get("aps_all") or {}
        segs    = res.get("segs") or []
        bad, total = [], 0
        for t, aps in aps_all.items():
            for ap in aps:
                total += 1
                if not any(s <= int(ap) <= e for s, e in segs):
                    bad.append((int(t), int(ap)))
        if bad:
            log(f"CP-5[{label}]", "FAIL",
                f"{len(bad)} APs outside segments (of {total}); first 3: {bad[:3]}")
        else:
            log(f"CP-5[{label}]", "PASS", f"all {total} APs inside segments")

    # ── CP-6: per-source stim frames live in their own part's range ─────────
    split  = int(meta.get("stim_split_local", 0) or 0)
    n_stim = int(meta.get("stim_n_samples", 0) or 0)
    if split == 0 or n_stim == 0:
        log("CP-6", "SKIP", "stim_split_local or stim_n_samples missing")
    else:
        bad = []
        for k, parts in sfps.items():
            if not isinstance(parts, dict):
                continue
            p1 = np.asarray(parts.get("part1", []) if parts.get("part1") is not None else [], dtype=int).ravel()
            p2 = np.asarray(parts.get("part2", []) if parts.get("part2") is not None else [], dtype=int).ravel()
            if p1.size and (p1.min() < 1 or p1.max() > split):
                bad.append((k, "part1", int(p1.min()), int(p1.max()), f"valid [1,{split}]"))
            if p2.size and (p2.min() < 1 or p2.max() > (n_stim - split)):
                bad.append((k, "part2", int(p2.min()), int(p2.max()), f"valid [1,{n_stim-split}]"))
        if bad:
            log("CP-6", "FAIL", f"{len(bad)} out-of-range entries; first: {bad[0]}")
        else:
            log("CP-6", "PASS",
                f"part1 ⊂ [1,{split}]  part2 ⊂ [1,{n_stim-split}]")

    # ── CP-7: merged_df pre/post IDs are all in all_traces ──────────────────
    merged = R.get("merged")
    if merged is not None and len(merged):
        all_t = set()
        for res in (scs, sta, R.get("scs2"), R.get("sta2")):
            if res:
                all_t |= set(int(x) for x in res.get("all_traces", []))
        try:
            bad_pre  = sorted(set(int(x) for x in merged["pre"])  - all_t)
            bad_post = sorted(set(int(x) for x in merged["post"]) - all_t)
        except Exception as e:
            log("CP-7", "FAIL", f"could not read pre/post columns: {e}")
        else:
            if bad_pre or bad_post:
                log("CP-7", "FAIL",
                    f"unknown pre={bad_pre[:10]}  unknown post={bad_post[:10]}")
            else:
                log("CP-7", "PASS",
                    f"{len(merged)} rows; all pre/post ∈ all_traces ({len(all_t)})")
    else:
        log("CP-7", "SKIP", "no merged table on R")

    # ── CP-8: stim-frame counts roughly match what each pipeline saw ────────
    # (visual raster check is interactive — see cp_stim_raster helper.)
    if sta and sfps:
        sta_active = sum(1 for t in sta.get("all_traces", [])
                         if sta.get("in_stim_mask", {}).get(t, False))
        log("CP-8", "PASS",
            f"{len(mask)} in stim_mask  |  {sta_active} flagged in STA in_stim_mask  |  "
            "for visual sanity call cp_stim_raster(meta)")
    else:
        log("CP-8", "SKIP", "no sta result or no per-source frames")

    # ── CP-9: round-trip one trace from CSV through detect_aps ──────────────
    if csv_paths and scs:
        ids = scs.get("all_traces", [])
        path = csv_paths.get("spont1") or csv_paths.get("spont2")
        if path and ids:
            try:
                from utils.data_utils import preprocess, detect_aps, detect_segments, regress_global_signal
                df   = pd.read_csv(path)
                df   = regress_global_signal(df)          # regress out global changes in the traces
                segs = detect_segments(df.iloc[:, 0].values)
                tid  = int(ids[0])
                raw  = df[f"T{tid}"].values.astype(float)
                aps_redo = detect_aps(preprocess(raw, segs), segs, raw=raw)
                n0, n1 = len(scs["aps_all"][tid]), len(aps_redo)
                if n0 == n1:
                    log("CP-9", "PASS", f"T{tid} APs round-trip ({n0})")
                else:
                    log("CP-9", "FAIL", f"T{tid} APs differ: stored={n0}  re-detect={n1}")
            except Exception as e:
                log("CP-9", "FAIL", f"{type(e).__name__}: {e}")
        else:
            log("CP-9", "SKIP", "no spont CSV or empty all_traces")
    else:
        log("CP-9", "SKIP", "no csv_paths or no scs result")

    # ── CP-10: SCS and STA neuron_type dicts cover the same trace universe ──
    if scs and sta:
        sset = set(int(k) for k in (scs.get("neuron_type") or {}))
        aset = set(int(k) for k in (sta.get("neuron_type") or {}))
        sym  = sorted(sset ^ aset)
        if sym:
            log("CP-10", "WARN",
                f"|scs_nt|={len(sset)} |sta_nt|={len(aset)} "
                f"symmetric_diff (first 10)={sym[:10]}")
        else:
            log("CP-10", "PASS",
                f"both neuron_type dicts cover {len(sset | aset)} IDs")
    else:
        log("CP-10", "SKIP", "need both scs and sta neuron_type")

    report = pd.DataFrame(rows, columns=["check", "status", "detail"])

    if verbose:
        n_pass = int((report["status"] == "PASS").sum())
        n_warn = int((report["status"] == "WARN").sum())
        n_fail = int((report["status"] == "FAIL").sum())
        n_skip = int((report["status"] == "SKIP").sum())
        print(f"\n  Summary: {n_pass} PASS · {n_warn} WARN · {n_fail} FAIL · {n_skip} SKIP")

    if strict and (report["status"] == "FAIL").any():
        fails = report[report["status"] == "FAIL"]
        raise AssertionError(
            f"{len(fails)} consistency check(s) failed for {fov}:\n"
            + fails.to_string(index=False)
        )

    return report

# ============================================================================
# Companion: visual stim raster for CP-8
# ============================================================================
def cp_stim_raster(meta, ax=None):
    """
    Per-source stim raster — orange dot for a source flagged in_stim_mask,
    red dot for one that has stim frames but is NOT in the mask (should be
    empty if the splitter is consistent).
    """
    import matplotlib.pyplot as plt

    mask = set(int(x) for x in meta.get("stim_mask_sources", []))
    split = int(meta.get("stim_split_local", 0) or 0)
    sfps  = meta.get("stim_frames_per_source", {}) or {}

    rows_in, cols_in, rows_out, cols_out = [], [], [], []
    for k, parts in sfps.items():
        if not isinstance(parts, dict):
            continue
        try:
            tid = int(str(k)[1:]) if str(k).startswith("T") else int(k)
        except ValueError:
            continue
        frames = []
        for f in (parts.get("part1") or []):
            frames.append(int(f))
        for f in (parts.get("part2") or []):
            frames.append(int(f) + split)
        if not frames:
            continue
        in_mask = tid in mask
        for fr in frames:
            (cols_in if in_mask else cols_out).append(fr)
            (rows_in if in_mask else rows_out).append(tid)

    if ax is None:
        _, ax = plt.subplots(figsize=(14, max(4, len(set(rows_in + rows_out)) * 0.12)))
    if cols_in:
        ax.scatter(cols_in, rows_in, s=4, c="tab:orange", label="in stim_mask")
    if cols_out:
        ax.scatter(cols_out, rows_out, s=8, c="tab:red",
                   label="HAS frames but NOT in mask (bug)")
    ax.set_xlabel("frame (concatenated stim block)")
    ax.set_ylabel("source ID")
    ax.set_title(f"{meta.get('fov','')} — stim raster sanity")
    ax.legend(loc="upper right", fontsize=8)
    return ax


In [ ]:
# Unit tests
if MODE == 'batch':
    def derive_csv_paths(R, input_dir):
        """
        Reconstruct the four CSV paths the splitter would have written for this FOV,
        using meta.fov + meta.savefast. Returns whichever files actually exist.
        """
        meta = R.get("meta") or {}
        fov  = meta.get("fov", "")
        sf   = meta.get("savefast", "")
        if not fov or not sf:
            return {}
        base = os.path.splitext(sf)[0]   # strip .savefast
        stem = f"{fov}_{base}"
        candidates = {
            "spont1": os.path.join(input_dir, f"{stem}_spont_part1.csv"),
            "spont2": os.path.join(input_dir, f"{stem}_spont_part2.csv"),
            "stim1":  os.path.join(input_dir, f"{stem}_stim_part1.csv"),
            "stim2":  os.path.join(input_dir, f"{stem}_stim_part2.csv"),
        }
        return {k: p for k, p in candidates.items() if os.path.exists(p)}
    all_reports = []
    for R in all_fov_results:
        csv_paths = derive_csv_paths(R, INPUT_DIR)
        rep = assert_pipeline_consistency(R, csv_paths=csv_paths, verbose=False)
        rep.insert(0, "fov", R.get("fov", ""))
        all_reports.append(rep)

    batch = pd.concat(all_reports, ignore_index=True)

    # All checks, all FOVs:
    display(batch)

    # Just the problems:
    issues = batch[batch["status"].isin(["WARN", "FAIL"])]
    print(f"\n{len(issues)} non-PASS rows across {batch['fov'].nunique()} FOVs:")
    print(issues.to_string(index=False) if len(issues) else "  (none)")

    # One-line per-FOV summary:
    summary = (batch.groupby("fov")["status"]
                    .value_counts().unstack(fill_value=0)
                    .reindex(columns=["PASS","WARN","FAIL","SKIP"], fill_value=0))
    display(summary)

## . Run Single FOV (when `MODE='single'`)

In [ ]:
if R is not None:
    all_traces = R['scs1']['all_traces'] if R['scs1'] else R['sta1']['all_traces']
    plot_consensus_heatmap(R['merged'], all_traces, output_path=os.path.join(R['fov_out'],
                     'connectionID_consensus_heatmap.png'))

In [ ]:
def _plot_stim_overlays(R, radius=4):
    fov         = R['fov']
    fov_dir_raw = os.path.join(INPUT_DIR.replace('v17_traces', ''), fov)  # raw FOV dir above trace CSVs

    # GFP (nuclear marker)
    plot_stim_neurons_on_gfp(R, fov_dir_raw, radius=radius,
        output_path=os.path.join(R['fov_out'], f'stimmed_neurons_gfp_{fov}.png'))

    # RCaMP (DLX marker — raw)
    _rcamp = os.path.join(fov_dir_raw, f'SupplementalImages_RCaMP_{fov}.png')
    if os.path.exists(_rcamp):
        plot_stim_neurons_on_gfp(R, fov_dir_raw, radius=radius, bg_path=_rcamp,
            output_path=os.path.join(R['fov_out'], f'stimmed_neurons_rcamp_{fov}.png'))

    # RCaMP enhanced (contrast-stretched, if present)
    _rcamp_enh = os.path.join(fov_dir_raw, f'SupplementalImages_RCaMP_{fov}_enhanced.png')
    if os.path.exists(_rcamp_enh):
        plot_stim_neurons_on_gfp(R, fov_dir_raw, radius=radius, bg_path=_rcamp_enh,
            output_path=os.path.join(R['fov_out'],
f'stimmed_neurons_rcamp_enhanced_{fov}.png'))

# Run for single FOV or every FOV in batch
_results = all_fov_results if MODE == 'batch' else ([R] if R is not None else [])
for _R in _results:
    if _R is not None:
        _plot_stim_overlays(_R)